In [1]:
!pip install -q transformers datasets accelerate scikit-learn openpyxl

In [2]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [4]:
FILE_PATH = "final_data_after_labling.xlsx"

df = pd.read_excel(FILE_PATH)
df.head()

,comment,is_toxic,threat,bullying,sexual_harassment,hate_speech,other_toxicity
0,absolute shit music and noise running in the b...,0,0,0,0,0,0
1,this game always puts you in a team with dumbass,0,0,0,0,0,0
2,constant server connection issues in the last ...,0,0,0,0,0,0
3,i picked this game up because a friend wanted ...,0,0,0,0,0,0
4,the new update destroyed the game rythm instea...,0,0,0,0,0,0


In [5]:
LABEL_COLS = [
    "is_toxic",
    "threat",
    "bullying",
    "sexual_harassment",
    "hate_speech",
    "other_toxicity"
]

TYPE_COLS = [
    "threat",
    "bullying",
    "sexual_harassment",
    "hate_speech",
    "other_toxicity"
]

# حذف الصفوف اللي التعليق فيها ناقص
df = df.dropna(subset=["comment"]).copy()

# تحويل النص إلى string
df["comment"] = df["comment"].astype(str).str.strip()

# حذف النصوص الفاضية
df = df[df["comment"] != ""].copy()

# التأكد أن الليبلز أرقام صحيحة
for col in LABEL_COLS:
    df[col] = df[col].astype(int)

print(df.shape)
print(df[LABEL_COLS].sum())

(11219, 7)
is_toxic             5987
threat               1620
bullying             2639
sexual_harassment    1620
hate_speech          2035
other_toxicity       2186
dtype: int64


In [6]:
train_df_1, val_df_1 = train_test_split(
    df[["comment", "is_toxic"]],
    test_size=0.2,
    random_state=42,
    stratify=df["is_toxic"]
)

train_df_1 = train_df_1.rename(columns={"is_toxic": "labels"})
val_df_1   = val_df_1.rename(columns={"is_toxic": "labels"})

In [7]:
MODEL_1_NAME = "distilbert-base-uncased"

tokenizer_1 = AutoTokenizer.from_pretrained(MODEL_1_NAME)
model_1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_1_NAME,
    num_labels=2
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
train_ds_1 = Dataset.from_pandas(train_df_1.reset_index(drop=True))
val_ds_1   = Dataset.from_pandas(val_df_1.reset_index(drop=True))

def tokenize_model1(batch):
    return tokenizer_1(
        batch["comment"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds_1 = train_ds_1.map(tokenize_model1, batched=True)
val_ds_1   = val_ds_1.map(tokenize_model1, batched=True)

train_ds_1.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds_1.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/8975 [00:00<?, ? examples/s]

Map:   0%|          | 0/2244 [00:00<?, ? examples/s]

In [9]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [10]:
MODEL_1_NAME = "distilbert-base-uncased"

tokenizer_1 = AutoTokenizer.from_pretrained(MODEL_1_NAME)

model_1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_1_NAME,
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
def compute_metrics_model1(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

args_1 = TrainingArguments(
    output_dir="./model_1_is_toxic",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer_1 = Trainer(
    model=model_1,
    args=args_1,
    train_dataset=train_ds_1,
    eval_dataset=val_ds_1,
    processing_class=tokenizer_1,
    compute_metrics=compute_metrics_model1
)

trainer_1.train()
trainer_1.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.112157,0.064894,0.977718,0.978992
2,0.054300,0.081263,0.977273,0.978616
3,0.031749,0.091076,0.977718,0.979097


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'eval_loss': 0.09107638150453568,
 'eval_accuracy': 0.9777183600713012,
 'eval_f1': 0.9790969899665551,
 'eval_runtime': 8.8266,
 'eval_samples_per_second': 254.233,
 'eval_steps_per_second': 15.975,
 'epoch': 3.0}

In [12]:
toxic_df = df[df["is_toxic"] == 1].copy()

# إذا فيه صفوف سامة لكن بدون أي نوع، نحذفها من تدريب المودل الثاني
toxic_df["type_sum"] = toxic_df[TYPE_COLS].sum(axis=1)
toxic_df = toxic_df[toxic_df["type_sum"] > 0].copy()

train_df_2, val_df_2 = train_test_split(
    toxic_df[["comment"] + TYPE_COLS],
    test_size=0.2,
    random_state=42
)

In [13]:
MODEL_2_NAME = "distilbert-base-uncased"

tokenizer_2 = AutoTokenizer.from_pretrained(MODEL_2_NAME)
model_2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_2_NAME,
    num_labels=len(TYPE_COLS),
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
train_ds_2 = Dataset.from_pandas(train_df_2.reset_index(drop=True))
val_ds_2   = Dataset.from_pandas(val_df_2.reset_index(drop=True))

def tokenize_model2(batch):
    return tokenizer_2(
        batch["comment"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds_2 = train_ds_2.map(tokenize_model2, batched=True)
val_ds_2   = val_ds_2.map(tokenize_model2, batched=True)

def format_multi_label(example):
    example["labels"] = [float(example[col]) for col in TYPE_COLS]
    return example

train_ds_2 = train_ds_2.map(format_multi_label)
val_ds_2   = val_ds_2.map(format_multi_label)

train_ds_2.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds_2.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/4782 [00:00<?, ? examples/s]

Map:   0%|          | 0/1196 [00:00<?, ? examples/s]

Map:   0%|          | 0/4782 [00:00<?, ? examples/s]

Map:   0%|          | 0/1196 [00:00<?, ? examples/s]

تدريب المودل الثاني

In [16]:
THRESHOLD = 0.5

def compute_metrics_model2(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= THRESHOLD).astype(int)

    exact_match = np.mean(np.all(preds == labels, axis=1))
    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)

    return {
        "exact_match": exact_match,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1
    }

args_2 = TrainingArguments(
    output_dir="./model_2_toxic_types",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    report_to="none"
)

trainer_2 = Trainer(
    model=model_2,
    args=args_2,
    train_dataset=train_ds_2,
    eval_dataset=val_ds_2,
    processing_class=tokenizer_2,
    compute_metrics=compute_metrics_model2
)

trainer_2.train()
trainer_2.evaluate()

Epoch,Training Loss,Validation Loss,Exact Match,Micro F1,Macro F1
1,0.416971,0.215337,0.795987,0.919486,0.917576
2,0.162305,0.120597,0.897157,0.962264,0.963340
3,0.108321,0.107599,0.904682,0.964550,0.965422


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'eval_loss': 0.10759939253330231,
 'eval_exact_match': 0.9046822742474916,
 'eval_micro_f1': 0.9645498597296608,
 'eval_macro_f1': 0.9654219151785256,
 'eval_runtime': 4.6503,
 'eval_samples_per_second': 257.185,
 'eval_steps_per_second': 16.128,
 'epoch': 3.0}

In [17]:
texts_1 = [
    "I have seen people in chat say some really messed up stuff, but it depends on the match honestly.",
    "Sometimes players get really aggressive, but I cannot tell if they are joking or being serious.",
    "A guy kept calling everyone useless, not sure if he was trolling or just mad.",
    "There was a moment where someone said something racist, but everyone ignored it and kept playing.",
    "The community can be rough, but I have also had some really good teammates.",
    "One player kept targeting me the whole game, I do not know if it was intentional or just how they play.",
    "People joke around a lot in voice chat, sometimes it goes too far though.",
    "I heard someone threaten another player, but it did not seem like they actually meant it.",
    "Some matches feel really toxic, others are completely fine, it is very inconsistent.",
    "I have seen players argue a lot, but I am not sure if it counts as toxicity or just competition."
]

true_labels_1 = np.array([
    [1,0,0,0,0,1],
    [1,0,0,0,0,1],
    [1,0,1,0,0,0],
    [1,0,0,0,1,0],
    [0,0,0,0,0,0],
    [0,0,0,0,0,0],
    [1,0,0,0,0,1],
    [1,1,0,0,0,0],
    [1,0,0,0,0,1],
    [0,0,0,0,0,0]
])

texts_2 = [
    "If you havent seen the toxicity that is the League of Legends community, I dont think you understand what the word means.",
    "My brother was telling me before they added Oceania servers to dota 2 he would have to play in asia and the asians would call him and his friends white devils.",
    "First time and only time I played with randos online I got yelled at for picking up the hand of Midas (I think that’s the name) as one of my first items, haven’t played online since and that was probably 4-5 years ago?",
    "My husband tried it when GTAV first came out and people just kept hunting him down over and over again and immediately killing him. He gave up on online after multiple attempts to actually do anything.",
    "Took my kid to a local smash mele tournament, which includes kids, and someone called him a terrorist and got banned from the store and I thought there was gonna be a fight when they told him to leave. Good thing smash trolls are soft and squishy.",
    "Not that bad actually, i would say just fix some horrible bugs",
    "Fck u developers and that stupid game",
    "its fine",
    "The Last of Us. Any small criticism and the fans will tell you lack media literacy while calling you a bigot. On the opposite side the game has toxic haters and sometimes they are homophobic too. So technically there is no place for valid criticism.",
    "The reputation of the LoL community basically put me off ever trying the genre."
]

true_labels_2 = np.array([
    [1,0,0,0,1,0],
    [1,0,0,0,0,1],
    [1,0,0,0,0,1],
    [0,0,0,0,0,0],
    [1,0,0,0,1,0],
    [0,0,0,0,0,0],
    [0,0,0,0,0,0],
    [0,0,0,0,0,0],
    [0,0,0,0,0,0],
    [1,0,0,0,0,1]
])

texts_3 = [
    "Every match in this game turns into people insulting the weakest player until they leave.",
    "I stopped using voice chat because some players kept threatening to find me and hurt me.",
    "The community is awful, they keep mocking new players for every small mistake.",
    "Some random guy in the lobby kept making disgusting sexual comments at me the whole game.",
    "I have seen players use racist slurs in chat whenever they get angry.",
    "The matchmaking is bad and the servers lag all the time.",
    "People in ranked keep telling others to uninstall and saying they are useless.",
    "One player said he would track me down if I queued against him again.",
    "The fanbase gets really defensive, but I would not call it toxic all the time.",
    "Women get harassed so much in this game's voice chat that many of them avoid speaking."
]

true_labels_3 = np.array([
    [1,0,1,0,0,0],
    [1,1,0,0,0,0],
    [1,0,1,0,0,0],
    [1,0,0,1,0,0],
    [1,0,0,0,1,0],
    [0,0,0,0,0,0],
    [1,0,1,0,0,0],
    [1,1,0,0,0,0],
    [0,0,0,0,0,0],
    [1,0,0,0,0,1]
])

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer_1.model.to(device)
trainer_2.model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [19]:
def predict_is_toxic(text):
    inputs = tokenizer_1(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = trainer_1.model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()

    return pred

In [20]:
def predict_toxic_types(text, threshold=0.5):
    inputs = tokenizer_2(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = trainer_2.model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]

    preds = (probs >= threshold).astype(int)
    return preds, probs

In [21]:
def pipeline_predict(text, threshold=0.5):
    pred_is_toxic = predict_is_toxic(text)

    if pred_is_toxic == 0:
        final_vec = np.array([0,0,0,0,0,0])
        probs = None
    else:
        type_preds, probs = predict_toxic_types(text, threshold=threshold)
        final_vec = np.concatenate([[1], type_preds])

    return final_vec, probs

التقييم النهائي على المجموعات ال3

In [22]:
def evaluate_pipeline_group(texts, true_labels, group_name, threshold=0.5):
    preds = []
    correct = 0

    print(f"\n{'='*60}")
    print(group_name)
    print(f"{'='*60}")

    for i, text in enumerate(texts):
        pred_vec, probs = pipeline_predict(text, threshold=threshold)
        preds.append(pred_vec)

        print("\n--------------------")
        print("TEXT:", text)
        print("TRUE:", true_labels[i].tolist())
        print("PRED:", pred_vec.tolist())

        if probs is not None:
            print("TYPE SCORES:", [round(x, 3) for x in probs.tolist()])

        if np.array_equal(pred_vec, true_labels[i]):
            print("✅ CORRECT")
            correct += 1
        else:
            print("❌ WRONG")

    preds = np.array(preds)
    exact_match_acc = np.mean(np.all(preds == true_labels, axis=1))

    print(f"\n{group_name} Correct: {correct}/{len(texts)}")
    print(f"{group_name} Exact Match Accuracy: {exact_match_acc:.2f}")

    return preds

In [23]:
preds_1 = evaluate_pipeline_group(texts_1, true_labels_1, "GROUP 1", threshold=0.5)
preds_2 = evaluate_pipeline_group(texts_2, true_labels_2, "GROUP 2", threshold=0.5)
preds_3 = evaluate_pipeline_group(texts_3, true_labels_3, "GROUP 3", threshold=0.5)


GROUP 1

--------------------
TEXT: I have seen people in chat say some really messed up stuff, but it depends on the match honestly.
TRUE: [1, 0, 0, 0, 0, 1]
PRED: [1, 0, 0, 0, 0, 0]
TYPE SCORES: [0.094, 0.027, 0.012, 0.031, 0.213]
❌ WRONG

--------------------
TEXT: Sometimes players get really aggressive, but I cannot tell if they are joking or being serious.
TRUE: [1, 0, 0, 0, 0, 1]
PRED: [1, 0, 0, 0, 0, 0]
TYPE SCORES: [0.265, 0.053, 0.026, 0.024, 0.045]
❌ WRONG

--------------------
TEXT: A guy kept calling everyone useless, not sure if he was trolling or just mad.
TRUE: [1, 0, 1, 0, 0, 0]
PRED: [1, 0, 0, 0, 0, 1]
TYPE SCORES: [0.034, 0.04, 0.134, 0.007, 0.805]
❌ WRONG

--------------------
TEXT: There was a moment where someone said something racist, but everyone ignored it and kept playing.
TRUE: [1, 0, 0, 0, 1, 0]
PRED: [1, 0, 0, 0, 1, 0]
TYPE SCORES: [0.047, 0.024, 0.025, 0.955, 0.032]
✅ CORRECT

--------------------
TEXT: The community can be rough, but I have also had some

النتيجه النهائية على ال30

In [24]:
all_texts = texts_1 + texts_2 + texts_3
all_true = np.vstack([true_labels_1, true_labels_2, true_labels_3])
all_preds = np.vstack([preds_1, preds_2, preds_3])

overall_exact = np.mean(np.all(all_preds == all_true, axis=1))
overall_correct = np.sum(np.all(all_preds == all_true, axis=1))
overall_is_toxic_acc = accuracy_score(all_true[:, 0], all_preds[:, 0])

print(f"\n{'='*70}")
print("FINAL RESULTS")
print(f"{'='*70}")
print(f"Overall Exact Match Accuracy: {overall_exact:.2f}")
print(f"Overall Correct: {overall_correct}/{len(all_true)}")
print(f"Overall is_toxic Accuracy: {overall_is_toxic_acc:.2f}")


FINAL RESULTS
Overall Exact Match Accuracy: 0.43
Overall Correct: 13/30
Overall is_toxic Accuracy: 0.67


In [25]:
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

# =========================
# LABELS
# =========================
TYPE_COLS = [
    "threat",
    "bullying",
    "sexual_harassment",
    "hate_speech",
    "other_toxicity"
]

ALL_COLS = ["is_toxic"] + TYPE_COLS

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trainer_1.model.to(device)
trainer_2.model.to(device)

# =========================
# PREDICTION FUNCTIONS
# =========================
def predict_is_toxic(text):
    inputs = tokenizer_1(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = trainer_1.model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()

    return pred

def predict_toxic_types(text, threshold=0.5):
    inputs = tokenizer_2(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = trainer_2.model(**inputs)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()[0]

    preds = (probs >= threshold).astype(int)
    return preds, probs

def pipeline_predict(text, threshold=0.5):
    pred_is_toxic = predict_is_toxic(text)

    if pred_is_toxic == 0:
        final_vec = np.array([0, 0, 0, 0, 0, 0])
        probs = np.array([0, 0, 0, 0, 0], dtype=float)
    else:
        type_preds, probs = predict_toxic_types(text, threshold=threshold)
        final_vec = np.concatenate([[1], type_preds])

    return final_vec, probs

# =========================
# EVALUATE ONE GROUP
# =========================
def evaluate_one_group(texts, true_labels, group_name="GROUP", threshold=0.5):
    rows = []
    preds = []

    print(f"\n{'='*70}")
    print(group_name)
    print(f"{'='*70}")

    for i, text in enumerate(texts):
        pred_vec, probs = pipeline_predict(text, threshold=threshold)
        true_vec = np.array(true_labels[i])

        preds.append(pred_vec)

        exact_match = int(np.array_equal(pred_vec, true_vec))

        row = {
            "comment": text,
            "true_is_toxic": true_vec[0],
            "true_threat": true_vec[1],
            "true_bullying": true_vec[2],
            "true_sexual_harassment": true_vec[3],
            "true_hate_speech": true_vec[4],
            "true_other_toxicity": true_vec[5],
            "pred_is_toxic": pred_vec[0],
            "pred_threat": pred_vec[1],
            "pred_bullying": pred_vec[2],
            "pred_sexual_harassment": pred_vec[3],
            "pred_hate_speech": pred_vec[4],
            "pred_other_toxicity": pred_vec[5],
            "exact_match": exact_match,
            "score_threat": round(float(probs[0]), 4),
            "score_bullying": round(float(probs[1]), 4),
            "score_sexual_harassment": round(float(probs[2]), 4),
            "score_hate_speech": round(float(probs[3]), 4),
            "score_other_toxicity": round(float(probs[4]), 4),
        }
        rows.append(row)

        print("\n--------------------")
        print("TEXT:", text)
        print("TRUE:", true_vec.tolist())
        print("PRED:", pred_vec.tolist())
        print("MATCH:", "✅" if exact_match else "❌")

    preds = np.array(preds)
    true_arr = np.array(true_labels)

    exact_acc = np.mean(np.all(preds == true_arr, axis=1))
    is_toxic_acc = accuracy_score(true_arr[:, 0], preds[:, 0])

    print(f"\n{group_name} Correct: {int(np.sum(np.all(preds == true_arr, axis=1)))}/{len(texts)}")
    print(f"{group_name} Exact Match Accuracy: {exact_acc:.2f}")
    print(f"{group_name} is_toxic Accuracy: {is_toxic_acc:.2f}")

    print(f"\n{group_name} Per-label Accuracy:")
    for j, col in enumerate(ALL_COLS):
        acc = accuracy_score(true_arr[:, j], preds[:, j])
        print(f"{col}: {acc:.2f}")

    results_df = pd.DataFrame(rows)
    return results_df, preds

In [27]:
group1_df, group1_preds = evaluate_one_group(texts_1, true_labels_1, group_name="GROUP 1", threshold=0.5)
group2_df, group2_preds = evaluate_one_group(texts_2, true_labels_2, group_name="GROUP 2", threshold=0.5)
group3_df, group3_preds = evaluate_one_group(texts_3, true_labels_3, group_name="GROUP 3", threshold=0.5)


GROUP 1

--------------------
TEXT: I have seen people in chat say some really messed up stuff, but it depends on the match honestly.
TRUE: [1, 0, 0, 0, 0, 1]
PRED: [1, 0, 0, 0, 0, 0]
MATCH: ❌

--------------------
TEXT: Sometimes players get really aggressive, but I cannot tell if they are joking or being serious.
TRUE: [1, 0, 0, 0, 0, 1]
PRED: [1, 0, 0, 0, 0, 0]
MATCH: ❌

--------------------
TEXT: A guy kept calling everyone useless, not sure if he was trolling or just mad.
TRUE: [1, 0, 1, 0, 0, 0]
PRED: [1, 0, 0, 0, 0, 1]
MATCH: ❌

--------------------
TEXT: There was a moment where someone said something racist, but everyone ignored it and kept playing.
TRUE: [1, 0, 0, 0, 1, 0]
PRED: [1, 0, 0, 0, 1, 0]
MATCH: ✅

--------------------
TEXT: The community can be rough, but I have also had some really good teammates.
TRUE: [0, 0, 0, 0, 0, 0]
PRED: [1, 0, 1, 0, 0, 0]
MATCH: ❌

--------------------
TEXT: One player kept targeting me the whole game, I do not know if it was intentional o

In [28]:
summary_df = pd.DataFrame([
    {
        "group": "GROUP 1",
        "correct": int(group1_df["exact_match"].sum()),
        "total": len(group1_df),
        "accuracy": round(group1_df["exact_match"].mean(), 4)
    },
    {
        "group": "GROUP 2",
        "correct": int(group2_df["exact_match"].sum()),
        "total": len(group2_df),
        "accuracy": round(group2_df["exact_match"].mean(), 4)
    },
    {
        "group": "GROUP 3",
        "correct": int(group3_df["exact_match"].sum()),
        "total": len(group3_df),
        "accuracy": round(group3_df["exact_match"].mean(), 4)
    }
])

print(summary_df)
summary_df.to_excel("groups_summary.xlsx", index=False)
print("Saved: groups_summary.xlsx")

     group  correct  total  accuracy
0  GROUP 1        3     10       0.3
1  GROUP 2        4     10       0.4
2  GROUP 3        6     10       0.6
Saved: groups_summary.xlsx
